In [9]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

from run_analysis import (
    load_and_validate,
    select_benchmark_sample,
    preprocess_features,
    run_detectors,
    calculate_metrics,
    calculate_overlaps,
    export_results,
)

DATA_PATH = Path("./data/creditcard.csv")
OUTPUT_DIR = Path("outputs")
MAX_ROWS = 30_000
DBSCAN_EPS_QUANTILE = 0.98
RANDOM_STATE = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
df, feature_cols = load_and_validate(DATA_PATH)

dataset_rows = len(df)
full_data_fraud_rate = float(df["Class"].mean())

print("Dataset shape:", df.shape)
print("Feature count:", len(feature_cols))
print("Full-data fraud rate:", f"{full_data_fraud_rate:.4%}")
print("Class distribution:")
display(df["Class"].value_counts().rename(index={0: "Legitimate", 1: "Fraud"}).to_frame("Count"))

[1/8] Loading and validating dataset...
  Rows: 284,807
  Numeric input features: 30
  Missing feature cells: 0
  Duplicate data rows (excluding generated ID): 1,081
  Class counts: {0: 284315, 1: 492}
Dataset shape: (284807, 32)
Feature count: 30
Full-data fraud rate: 0.1727%
Class distribution:


,Count
Class,
Legitimate,284315
Fraud,492


In [3]:
sample = select_benchmark_sample(df, MAX_ROWS)

print("Benchmark shape:", sample.shape)
print("Benchmark fraud rate:", f"{sample['Class'].mean():.4%}")
display(sample[feature_cols + ["Class"]].head())

[2/8] Selecting a shared benchmark sample...
  Selected 30,000 rows (10.5% of the dataset).
  Stratification is used only to preserve class proportions in the evaluation sample.
  Sample class counts: {0: 29948, 1: 52}
Benchmark shape: (30000, 32)
Benchmark fraud rate: 0.1733%


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,4.0,1.229658,0.141004,0.045371,1.202613,0.191881,0.272708,-0.005159,0.081213,0.464960,...,-0.167716,-0.270710,-0.154104,-0.780055,0.750137,-0.257237,0.034507,0.005168,4.99,0
2,10.0,1.249999,-1.221637,0.383930,-1.234899,-1.485419,-0.753230,-0.689405,-0.227487,-2.094011,...,-0.231809,-0.483285,0.084668,0.392831,0.161135,-0.354990,0.026416,0.042422,121.50,0
3,12.0,1.103215,-0.040296,1.267332,1.289091,-0.735997,0.288069,-0.586057,0.189380,0.782333,...,-0.024612,0.196002,0.013802,0.103758,0.364298,-0.382261,0.092809,0.037051,12.99,0
4,22.0,-1.946525,-0.044901,-0.405570,-1.013057,2.941968,2.955053,-0.063063,0.855546,0.049967,...,-0.579526,-0.799229,0.870300,0.983421,0.321201,0.149650,0.707519,0.014600,0.89,0


In [4]:
X = preprocess_features(sample, feature_cols)
y = sample["Class"].astype(int).to_numpy()
transaction_ids = sample["Transaction_ID"].astype(int).to_numpy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Fraud cases in benchmark:", int(y.sum()))

[3/8] Preprocessing features...
  X shape: (30000, 30)
  Used: log1p(Amount), SimpleImputer(median), StandardScaler.
X shape: (30000, 30)
y shape: (30000,)
Fraud cases in benchmark: 52


In [5]:
scores, predictions, metadata = run_detectors(
    X=X,
    transaction_ids=transaction_ids,
    fraud_rate=full_data_fraud_rate,
    dbscan_eps_quantile=DBSCAN_EPS_QUANTILE,
)

print("Methods:", list(predictions.keys()))
print("Top-k threshold budget:", metadata["expected_k"])
print("DBSCAN eps:", metadata["dbscan_eps"])
print("DBSCAN noise points:", metadata["dbscan_noise_count"])

[4/8] Fitting scikit-learn outlier detectors...
  Shared alert budget for score-based methods: 52 rows.
  - Mahalanobis Distance: sklearn.covariance.LedoitWolf
  - Isolation Forest: sklearn.ensemble.IsolationForest
  - Local Outlier Factor: sklearn.neighbors.LocalOutlierFactor
  - DBSCAN: sklearn.cluster.DBSCAN
  - PCA Reconstruction Error: sklearn.decomposition.PCA
  - kNN distance: sklearn.neighbors.NearestNeighbors
Methods: ['Mahalanobis', 'Isolation Forest', 'LOF', 'DBSCAN', 'PCA Reconstruction', 'kNN Distance']
Top-k threshold budget: 52
DBSCAN eps: 3.681932482030481
DBSCAN noise points: 309


In [6]:
metrics = calculate_metrics(y, predictions, scores)
display(metrics.round(4))

[5/8] Calculating evaluation metrics...


,Method,Detected_Outliers,True_Positives,False_Positives,False_Negatives,True_Negatives,Precision,Recall,F1,Specificity,Balanced_Accuracy,ROC_AUC,Average_Precision
0,kNN Distance,52,20,32,32,29916,0.3846,0.3846,0.3846,0.9989,0.6918,0.9671,0.3340
1,Mahalanobis,52,16,36,36,29912,0.3077,0.3077,0.3077,0.9988,0.6532,0.9481,0.2188
2,Isolation Forest,52,15,37,37,29911,0.2885,0.2885,0.2885,0.9988,0.6436,0.9414,0.2345
3,DBSCAN,309,44,265,8,29683,0.1424,0.8462,0.2438,0.9912,0.9187,NaN,NaN
4,LOF,52,7,45,45,29903,0.1346,0.1346,0.1346,0.9985,0.5666,0.9251,0.0979
5,PCA Reconstruction,52,0,52,52,29896,0.0000,0.0000,0.0000,0.9983,0.4991,0.5452,0.0023


In [7]:
overlap_counts, consensus, common_all_ids, common = calculate_overlaps(
    transaction_ids,
    predictions,
    y_true=y,
)
jaccard = overlap_counts.attrs["jaccard"]

print("Pairwise intersection counts:")
display(overlap_counts)
print("Pairwise Jaccard similarities:")
display(jaccard.round(4))
print(f"Rows flagged by all {len(predictions)} methods: {len(common)}")
print("Common Transaction_ID values:", sorted(common_all_ids))
print("Rows flagged by at least 3 methods:", len(consensus))

display(common.head(50))
display(consensus.head(20))

[6/8] Comparing detected rows and calculating intersections...
Pairwise intersection counts:


,Mahalanobis,Isolation Forest,LOF,DBSCAN,PCA Reconstruction,kNN Distance
Mahalanobis,52.0,44.0,24.0,49.0,13.0,44.0
Isolation Forest,44.0,52.0,21.0,48.0,13.0,41.0
LOF,24.0,21.0,52.0,34.0,13.0,22.0
DBSCAN,49.0,48.0,34.0,309.0,19.0,52.0
PCA Reconstruction,13.0,13.0,13.0,19.0,52.0,13.0
kNN Distance,44.0,41.0,22.0,52.0,13.0,52.0


Pairwise Jaccard similarities:


,Mahalanobis,Isolation Forest,LOF,DBSCAN,PCA Reconstruction,kNN Distance
Mahalanobis,1.0000,0.7333,0.3000,0.1571,0.1429,0.7333
Isolation Forest,0.7333,1.0000,0.2530,0.1534,0.1429,0.6508
LOF,0.3000,0.2530,1.0000,0.1040,0.1429,0.2683
DBSCAN,0.1571,0.1534,0.1040,1.0000,0.0556,0.1683
PCA Reconstruction,0.1429,0.1429,0.1429,0.0556,1.0000,0.1429
kNN Distance,0.7333,0.6508,0.2683,0.1683,0.1429,1.0000


Rows flagged by all 6 methods: 11
Common Transaction_ID values: [48529, 76679, 89606, 117639, 213160, 224569, 226690, 238412, 239532, 244865, 261843]
Rows flagged by at least 3 methods: 51


,Transaction_ID,Class,Pred_Mahalanobis,Pred_Isolation_Forest,Pred_LOF,Pred_DBSCAN,Pred_PCA_Reconstruction,Pred_kNN_Distance,Consensus_Count
5124,48529,0,1,1,1,1,1,1,6
8050,76679,0,1,1,1,1,1,1,6
9445,89606,0,1,1,1,1,1,1,6
12361,117639,0,1,1,1,1,1,1,6
22482,213160,0,1,1,1,1,1,1,6
23596,224569,0,1,1,1,1,1,1,6
23833,226690,0,1,1,1,1,1,1,6
25059,238412,0,1,1,1,1,1,1,6
25189,239532,0,1,1,1,1,1,1,6
25782,244865,0,1,1,1,1,1,1,6


,Transaction_ID,Class,Pred_Mahalanobis,Pred_Isolation_Forest,Pred_LOF,Pred_DBSCAN,Pred_PCA_Reconstruction,Pred_kNN_Distance,Consensus_Count
5124,48529,0,1,1,1,1,1,1,6
8050,76679,0,1,1,1,1,1,1,6
9445,89606,0,1,1,1,1,1,1,6
12361,117639,0,1,1,1,1,1,1,6
22482,213160,0,1,1,1,1,1,1,6
23596,224569,0,1,1,1,1,1,1,6
23833,226690,0,1,1,1,1,1,1,6
25059,238412,0,1,1,1,1,1,1,6
25189,239532,0,1,1,1,1,1,1,6
25782,244865,0,1,1,1,1,1,1,6


In [8]:
export_results(
    output_dir=OUTPUT_DIR,
    data_path=DATA_PATH,
    dataset_rows=dataset_rows,
    fraud_rate=full_data_fraud_rate,
    sample=sample,
    feature_columns=feature_cols,
    y=y,
    transaction_ids=transaction_ids,
    scores=scores,
    predictions=predictions,
    metrics=metrics,
    overlap_counts=overlap_counts,
    consensus=consensus,
    common_ids=common_all_ids,
    common=common,
    metadata=metadata,
)

print("Saved outputs to:", OUTPUT_DIR.resolve())
print("Main report:", OUTPUT_DIR / "results_report.md")
print("All-method intersection:", OUTPUT_DIR / "common_outliers_all_methods.csv")

display(metrics.round(4))
display(common)
display(consensus.head(30))

[7/8] Generating plots...
Saved outputs to: C:\Users\Saeed.kh\Desktop\HW-1\outputs
Main report: outputs\results_report.md
All-method intersection: outputs\common_outliers_all_methods.csv


,Method,Detected_Outliers,True_Positives,False_Positives,False_Negatives,True_Negatives,Precision,Recall,F1,Specificity,Balanced_Accuracy,ROC_AUC,Average_Precision
0,kNN Distance,52,20,32,32,29916,0.3846,0.3846,0.3846,0.9989,0.6918,0.9671,0.3340
1,Mahalanobis,52,16,36,36,29912,0.3077,0.3077,0.3077,0.9988,0.6532,0.9481,0.2188
2,Isolation Forest,52,15,37,37,29911,0.2885,0.2885,0.2885,0.9988,0.6436,0.9414,0.2345
3,DBSCAN,309,44,265,8,29683,0.1424,0.8462,0.2438,0.9912,0.9187,NaN,NaN
4,LOF,52,7,45,45,29903,0.1346,0.1346,0.1346,0.9985,0.5666,0.9251,0.0979
5,PCA Reconstruction,52,0,52,52,29896,0.0000,0.0000,0.0000,0.9983,0.4991,0.5452,0.0023


,Transaction_ID,Class,Pred_Mahalanobis,Pred_Isolation_Forest,Pred_LOF,Pred_DBSCAN,Pred_PCA_Reconstruction,Pred_kNN_Distance,Consensus_Count
5124,48529,0,1,1,1,1,1,1,6
8050,76679,0,1,1,1,1,1,1,6
9445,89606,0,1,1,1,1,1,1,6
12361,117639,0,1,1,1,1,1,1,6
22482,213160,0,1,1,1,1,1,1,6
23596,224569,0,1,1,1,1,1,1,6
23833,226690,0,1,1,1,1,1,1,6
25059,238412,0,1,1,1,1,1,1,6
25189,239532,0,1,1,1,1,1,1,6
25782,244865,0,1,1,1,1,1,1,6


,Transaction_ID,Class,Pred_Mahalanobis,Pred_Isolation_Forest,Pred_LOF,Pred_DBSCAN,Pred_PCA_Reconstruction,Pred_kNN_Distance,Consensus_Count
5124,48529,0,1,1,1,1,1,1,6
8050,76679,0,1,1,1,1,1,1,6
9445,89606,0,1,1,1,1,1,1,6
12361,117639,0,1,1,1,1,1,1,6
22482,213160,0,1,1,1,1,1,1,6
23596,224569,0,1,1,1,1,1,1,6
23833,226690,0,1,1,1,1,1,1,6
25059,238412,0,1,1,1,1,1,1,6
25189,239532,0,1,1,1,1,1,1,6
25782,244865,0,1,1,1,1,1,1,6
